# CodeGen0.4

Notebook ini fokus pada **error analysis** hasil `Gen0.2`, terutama untuk region seperti `Jawa` yang masih menunjukkan indikasi miss besar pada sebagian sampel.


In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


DATA_PATH = Path("nasa_dataset_daily_20200101_20260101_representative_point_area.csv")
ARTIFACTS_ROOT_CANDIDATES = [
    Path("artifacts_xgboost_representative_points(Gen0.2)"),
    Path("artifacts_xgboost_representative_points"),
    Path("artifacts_xgboost_representative_points_Gen0.2"),
]
OUTPUT_ROOT = Path("artifacts_error_analysis(Gen0.4)")

TARGET_COL = "ALLSKY_SFC_SW_DWN"
MISSING_SENTINEL = -999.0
LOOKBACK = 24
HORIZON = 24
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
MAPE_EPSILON = 1.0

BASE_FEATURES = [
    "CLOUD_AMT",
    "T2M",
    "RH2M",
    "PS",
    "CLRSKY_SFC_SW_DWN",
]

REGION_NAME_MAP = {
    "sumatra": "Sumatra",
    "sumatera": "Sumatra",
    "jawa": "Jawa",
    "kalimantan": "Kalimantan",
    "sulawesi": "Sulawesi",
    "nusa tenggara": "Nusa Tenggara",
    "bali_nusa": "Nusa Tenggara",
    "ntb_ntt": "Nusa Tenggara",
    "maluku": "Maluku",
    "papua": "Papua",
}




## Data Prep dan Reload Prediction


In [2]:
def normalize_region_name(name: str) -> str | None:
    if pd.isna(name):
        return None
    return REGION_NAME_MAP.get(str(name).strip().lower())


def region_to_folder_name(region_name: str) -> str:
    return region_name.replace(" ", "_")


def resolve_artifacts_root() -> Path:
    for candidate in ARTIFACTS_ROOT_CANDIDATES:
        if candidate.exists():
            print(f"[INFO] Using artifacts root: {candidate}")
            return candidate

    checked = ", ".join(str(path) for path in ARTIFACTS_ROOT_CANDIDATES)
    raise FileNotFoundError(
        "Could not find Gen0.2 artifacts root. "
        f"Checked: {checked}"
    )


def load_dataset(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df["REP_NAME"] = df["REP_NAME"].map(normalize_region_name)
    df["datetime"] = pd.to_datetime(
        {
            "year": df["YEAR"],
            "month": df["MO"],
            "day": df["DY"],
            "hour": df["HR"],
        },
        errors="coerce",
    )
    df = df.dropna(subset=["datetime", "REP_NAME"]).copy()

    numeric_cols = BASE_FEATURES + [TARGET_COL]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df.loc[df[col] == MISSING_SENTINEL, col] = np.nan

    return df.sort_values(["REP_NAME", "datetime"]).reset_index(drop=True)


def create_time_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["sin_hour"] = np.sin(2 * np.pi * out["HR"] / 24.0)
    out["cos_hour"] = np.cos(2 * np.pi * out["HR"] / 24.0)
    out["sin_month"] = np.sin(2 * np.pi * out["MO"] / 12.0)
    out["cos_month"] = np.cos(2 * np.pi * out["MO"] / 12.0)
    return out


def create_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for lag in range(1, LOOKBACK + 1):
        out[f"{TARGET_COL}_lag_{lag}"] = out[TARGET_COL].shift(lag)

    for feature in BASE_FEATURES:
        for lag in [1, 2, 3, 6, 12, 24]:
            out[f"{feature}_lag_{lag}"] = out[feature].shift(lag)

    for feature in [TARGET_COL, "CLOUD_AMT", "CLRSKY_SFC_SW_DWN"]:
        shifted = out[feature].shift(1)
        out[f"{feature}_roll_mean_3"] = shifted.rolling(window=3, min_periods=3).mean()
        out[f"{feature}_roll_mean_6"] = shifted.rolling(window=6, min_periods=6).mean()
        out[f"{feature}_roll_min_6"] = shifted.rolling(window=6, min_periods=6).min()
        out[f"{feature}_roll_max_6"] = shifted.rolling(window=6, min_periods=6).max()

    return out


def create_multistep_targets(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for step in range(1, HORIZON + 1):
        out[f"target_t_plus_{step}"] = out[TARGET_COL].shift(-step)
    out["forecast_start"] = out["datetime"].shift(-1)
    return out


def prepare_supervised_region_data(df: pd.DataFrame, region_name: str) -> tuple[pd.DataFrame, list[str], list[str]]:
    region_df = df.loc[df["REP_NAME"] == region_name].copy()
    region_df = region_df.dropna(subset=[TARGET_COL]).copy()
    region_df[BASE_FEATURES] = region_df[BASE_FEATURES].ffill()

    processed = create_time_features(region_df)
    processed = create_lag_features(processed)
    processed = create_multistep_targets(processed)

    target_cols = [f"target_t_plus_{step}" for step in range(1, HORIZON + 1)]
    feature_cols = [
        "CLOUD_AMT",
        "T2M",
        "RH2M",
        "PS",
        "CLRSKY_SFC_SW_DWN",
        "sin_hour",
        "cos_hour",
        "sin_month",
        "cos_month",
    ]
    feature_cols += [
        col
        for col in processed.columns
        if col.startswith(f"{TARGET_COL}_lag_")
        or any(col.startswith(f"{feature}_lag_") for feature in BASE_FEATURES)
        or "_roll_" in col
    ]

    required_history_cols = [f"{TARGET_COL}_lag_{LOOKBACK}"] + [f"{feature}_lag_24" for feature in BASE_FEATURES]
    processed = processed.dropna(subset=required_history_cols + ["forecast_start"]).copy()
    processed = processed.dropna(subset=target_cols).copy()
    processed = processed.sort_values("datetime").reset_index(drop=True)
    return processed, feature_cols, target_cols


def time_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    n_rows = len(df)
    train_end = int(n_rows * TRAIN_RATIO)
    val_end = int(n_rows * (TRAIN_RATIO + VAL_RATIO))
    return (
        df.iloc[:train_end].copy(),
        df.iloc[train_end:val_end].copy(),
        df.iloc[val_end:].copy(),
    )


def apply_train_median_imputation(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    medians = train_df[feature_cols].median(numeric_only=True).to_dict()
    train_df[feature_cols] = train_df[feature_cols].fillna(medians)
    val_df[feature_cols] = val_df[feature_cols].fillna(medians)
    test_df[feature_cols] = test_df[feature_cols].fillna(medians)
    return train_df, val_df, test_df


def load_region_artifacts(region_name: str) -> tuple[list[Any], dict[str, Any], dict[str, Any]]:
    artifacts_root = resolve_artifacts_root()
    region_dir = artifacts_root / region_to_folder_name(region_name)
    models = joblib.load(region_dir / "xgboost_models.joblib")
    preprocessor = joblib.load(region_dir / "preprocessor.joblib")
    metrics = json.loads((region_dir / "metrics.json").read_text(encoding="utf-8"))
    return models, preprocessor, metrics


def generate_predictions(region_name: str) -> dict[str, Any]:
    df = load_dataset(DATA_PATH)
    supervised_df, feature_cols, target_cols = prepare_supervised_region_data(df, region_name)
    train_df, val_df, test_df = time_split(supervised_df)
    train_df, val_df, test_df = apply_train_median_imputation(train_df, val_df, test_df, feature_cols)

    models, preprocessor, saved_metrics = load_region_artifacts(region_name)
    x_test = test_df[feature_cols].to_numpy(dtype=np.float32)
    y_test = test_df[target_cols].to_numpy(dtype=np.float32)
    y_pred = np.column_stack([model.predict(x_test) for model in models]).astype(np.float32)

    post_cfg = preprocessor.get("post_processing", {})
    if post_cfg.get("clip_non_negative", True):
        y_pred = np.clip(y_pred, 0, None)

    return {
        "test_df": test_df,
        "feature_cols": feature_cols,
        "target_cols": target_cols,
        "y_true": y_test,
        "y_pred": y_pred,
        "saved_metrics": saved_metrics,
    }




## Error Metrics dan Breakdown


In [3]:
def compute_global_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    flat_true = y_true.reshape(-1)
    flat_pred = y_pred.reshape(-1)
    abs_err = np.abs(flat_true - flat_pred)
    mask = np.abs(flat_true) > MAPE_EPSILON
    mape = float(np.mean(np.abs((flat_true[mask] - flat_pred[mask]) / flat_true[mask])) * 100.0) if mask.any() else float("nan")
    return {
        "rmse": float(np.sqrt(mean_squared_error(flat_true, flat_pred))),
        "mae": float(mean_absolute_error(flat_true, flat_pred)),
        "r2": float(r2_score(flat_true, flat_pred)),
        "mape_pct": mape,
        "p90_abs_error": float(np.percentile(abs_err, 90)),
        "p95_abs_error": float(np.percentile(abs_err, 95)),
        "p99_abs_error": float(np.percentile(abs_err, 99)),
        "max_abs_error": float(np.max(abs_err)),
    }


def build_error_frame(test_df: pd.DataFrame, y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    rows = []
    for sample_idx in range(len(test_df)):
        start_ts = pd.to_datetime(test_df.iloc[sample_idx]["forecast_start"])
        for horizon_idx in range(HORIZON):
            ts = start_ts + pd.Timedelta(hours=horizon_idx)
            actual = float(y_true[sample_idx, horizon_idx])
            pred = float(y_pred[sample_idx, horizon_idx])
            abs_err = abs(actual - pred)
            ape = np.nan
            if abs(actual) > MAPE_EPSILON:
                ape = abs(actual - pred) / abs(actual) * 100.0
            rows.append(
                {
                    "sample_idx": sample_idx,
                    "forecast_start": start_ts,
                    "timestamp": ts,
                    "hour": ts.hour,
                    "horizon": horizon_idx + 1,
                    "actual": actual,
                    "predicted": pred,
                    "error": pred - actual,
                    "abs_error": abs_err,
                    "ape_pct": ape,
                    "is_daytime": 6 <= ts.hour <= 17,
                }
            )
    return pd.DataFrame(rows)


def summarize_by_hour(error_df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        error_df.groupby("hour")
        .agg(
            rmse=("error", lambda x: float(np.sqrt(np.mean(np.square(x))))),
            mae=("abs_error", "mean"),
            mape_pct=("ape_pct", "mean"),
            p95_abs_error=("abs_error", lambda x: float(np.percentile(x, 95))),
            count=("abs_error", "size"),
        )
        .reset_index()
    )
    return summary


def summarize_by_horizon(error_df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        error_df.groupby("horizon")
        .agg(
            rmse=("error", lambda x: float(np.sqrt(np.mean(np.square(x))))),
            mae=("abs_error", "mean"),
            mape_pct=("ape_pct", "mean"),
            p95_abs_error=("abs_error", lambda x: float(np.percentile(x, 95))),
            count=("abs_error", "size"),
        )
        .reset_index()
    )
    return summary


def summarize_day_night(error_df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        error_df.assign(period=np.where(error_df["is_daytime"], "daytime", "nighttime"))
        .groupby("period")
        .agg(
            rmse=("error", lambda x: float(np.sqrt(np.mean(np.square(x))))),
            mae=("abs_error", "mean"),
            mape_pct=("ape_pct", "mean"),
            p95_abs_error=("abs_error", lambda x: float(np.percentile(x, 95))),
            count=("abs_error", "size"),
        )
        .reset_index()
    )
    return summary


def summarize_actual_bins(error_df: pd.DataFrame) -> pd.DataFrame:
    bins = [-0.1, 1.0, 50.0, 150.0, 300.0, 600.0, 2000.0]
    labels = ["0-1", "1-50", "50-150", "150-300", "300-600", "600+"]
    temp = error_df.copy()
    temp["actual_bin"] = pd.cut(temp["actual"], bins=bins, labels=labels)
    summary = (
        temp.groupby("actual_bin", observed=False)
        .agg(
            rmse=("error", lambda x: float(np.sqrt(np.mean(np.square(x)))) if len(x) else np.nan),
            mae=("abs_error", "mean"),
            mape_pct=("ape_pct", "mean"),
            p95_abs_error=("abs_error", lambda x: float(np.percentile(x, 95)) if len(x) else np.nan),
            count=("abs_error", "size"),
        )
        .reset_index()
    )
    return summary


def plot_error_by_hour(summary_df: pd.DataFrame, save_path: Path) -> None:
    plt.figure(figsize=(12, 5))
    plt.plot(summary_df["hour"], summary_df["rmse"], marker="o", label="RMSE")
    plt.plot(summary_df["hour"], summary_df["mae"], marker="o", label="MAE")
    plt.title("Error by Forecast Hour")
    plt.xlabel("Hour of day")
    plt.ylabel("Error")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def plot_error_by_horizon(summary_df: pd.DataFrame, save_path: Path) -> None:
    plt.figure(figsize=(12, 5))
    plt.plot(summary_df["horizon"], summary_df["rmse"], marker="o", label="RMSE")
    plt.plot(summary_df["horizon"], summary_df["mae"], marker="o", label="MAE")
    plt.title("Error by Horizon")
    plt.xlabel("Horizon")
    plt.ylabel("Error")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def plot_abs_error_distribution(error_df: pd.DataFrame, save_path: Path) -> None:
    plt.figure(figsize=(10, 5))
    plt.hist(error_df["abs_error"], bins=80)
    plt.title("Absolute Error Distribution")
    plt.xlabel("Absolute error")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def plot_actual_vs_error(error_df: pd.DataFrame, save_path: Path) -> None:
    sample_df = error_df.sample(min(4000, len(error_df)), random_state=42)
    plt.figure(figsize=(10, 6))
    plt.scatter(sample_df["actual"], sample_df["abs_error"], s=10, alpha=0.3)
    plt.title("Actual vs Absolute Error")
    plt.xlabel("Actual irradiance")
    plt.ylabel("Absolute error")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()




## Save Outputs dan Run


In [4]:
def save_analysis_outputs(region_name: str, analysis: dict[str, Any]) -> dict[str, str]:
    region_dir = OUTPUT_ROOT / region_to_folder_name(region_name)
    region_dir.mkdir(parents=True, exist_ok=True)

    global_metrics_path = region_dir / "global_metrics.json"
    top_errors_path = region_dir / "top_100_errors.csv"
    error_frame_path = region_dir / "error_frame_sample_5000.csv"
    hour_summary_path = region_dir / "error_by_hour.csv"
    horizon_summary_path = region_dir / "error_by_horizon.csv"
    daynight_summary_path = region_dir / "error_day_vs_night.csv"
    actual_bins_path = region_dir / "error_by_actual_bin.csv"
    summary_txt_path = region_dir / "analysis_summary.txt"

    hour_plot_path = region_dir / "error_by_hour.png"
    horizon_plot_path = region_dir / "error_by_horizon.png"
    dist_plot_path = region_dir / "abs_error_distribution.png"
    actual_error_plot_path = region_dir / "actual_vs_abs_error.png"

    with global_metrics_path.open("w", encoding="utf-8") as f:
        json.dump(analysis["global_metrics"], f, indent=2)

    analysis["top_errors"].to_csv(top_errors_path, index=False)
    analysis["error_df"].head(5000).to_csv(error_frame_path, index=False)
    analysis["hour_summary"].to_csv(hour_summary_path, index=False)
    analysis["horizon_summary"].to_csv(horizon_summary_path, index=False)
    analysis["day_night_summary"].to_csv(daynight_summary_path, index=False)
    analysis["actual_bin_summary"].to_csv(actual_bins_path, index=False)

    plot_error_by_hour(analysis["hour_summary"], hour_plot_path)
    plot_error_by_horizon(analysis["horizon_summary"], horizon_plot_path)
    plot_abs_error_distribution(analysis["error_df"], dist_plot_path)
    plot_actual_vs_error(analysis["error_df"], actual_error_plot_path)

    gm = analysis["global_metrics"]
    worst_hour = analysis["hour_summary"].sort_values("rmse", ascending=False).iloc[0]
    worst_horizon = analysis["horizon_summary"].sort_values("rmse", ascending=False).iloc[0]

    summary_lines = [
        f"Region             : {region_name}",
        f"Artifacts source   : {(resolve_artifacts_root() / region_to_folder_name(region_name)).resolve()}",
        f"Analysis dir       : {region_dir.resolve()}",
        f"RMSE               : {gm['rmse']:.6f}",
        f"MAE                : {gm['mae']:.6f}",
        f"R2                 : {gm['r2']:.6f}",
        f"MAPE               : {gm['mape_pct']:.6f}%",
        f"P95 abs error      : {gm['p95_abs_error']:.6f}",
        f"P99 abs error      : {gm['p99_abs_error']:.6f}",
        f"Max abs error      : {gm['max_abs_error']:.6f}",
        f"Worst hour         : {int(worst_hour['hour'])} (RMSE={worst_hour['rmse']:.6f})",
        f"Worst horizon      : {int(worst_horizon['horizon'])} (RMSE={worst_horizon['rmse']:.6f})",
    ]
    summary_txt_path.write_text("\n".join(summary_lines), encoding="utf-8")

    return {
        "region_dir": str(region_dir.resolve()),
        "summary_txt_path": str(summary_txt_path.resolve()),
        "top_errors_path": str(top_errors_path.resolve()),
    }


def analyze_region(region_name: str = "Jawa") -> dict[str, Any]:
    print(f"[INFO] Running error analysis for region: {region_name}")
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    bundle = generate_predictions(region_name)
    error_df = build_error_frame(bundle["test_df"], bundle["y_true"], bundle["y_pred"])

    analysis = {
        "global_metrics": compute_global_metrics(bundle["y_true"], bundle["y_pred"]),
        "error_df": error_df,
        "hour_summary": summarize_by_hour(error_df),
        "horizon_summary": summarize_by_horizon(error_df),
        "day_night_summary": summarize_day_night(error_df),
        "actual_bin_summary": summarize_actual_bins(error_df),
        "top_errors": error_df.sort_values("abs_error", ascending=False).head(100).reset_index(drop=True),
        "saved_metrics": bundle["saved_metrics"],
    }

    paths = save_analysis_outputs(region_name, analysis)

    print("[INFO] Global metrics:")
    print(json.dumps(analysis["global_metrics"], indent=2))
    print("[INFO] Day vs night summary:")
    print(analysis["day_night_summary"].to_string(index=False))
    print("[INFO] Top 10 biggest errors:")
    print(analysis["top_errors"].head(10).to_string(index=False))
    print(f"[INFO] Analysis outputs saved to: {paths['region_dir']}")

    return {
        "region_name": region_name,
        "analysis_dir": paths["region_dir"],
        "global_metrics": analysis["global_metrics"],
        "hour_summary": analysis["hour_summary"],
        "horizon_summary": analysis["horizon_summary"],
        "day_night_summary": analysis["day_night_summary"],
        "top_errors": analysis["top_errors"],
    }


def main() -> None:
    result = analyze_region(region_name="Jawa")
    print(f"[INFO] Done. Analysis dir: {result['analysis_dir']}")


if __name__ == "__main__":
    main()


[INFO] Running error analysis for region: Jawa
[INFO] Using artifacts root: artifacts_xgboost_representative_points


/home/dawwi/AI-Model/venv/lib/python3.13/site-packages/xgboost/core.py:158: UserWarning: [11:23:36] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


[INFO] Using artifacts root: artifacts_xgboost_representative_points
[INFO] Global metrics:
{
  "rmse": 61.721920013427734,
  "mae": 29.90321922302246,
  "r2": 0.9464049935340881,
  "mape_pct": 23.364572525024414,
  "p90_abs_error": 100.4588623046875,
  "p95_abs_error": 144.45993041992188,
  "p99_abs_error": 245.39511108398438,
  "max_abs_error": 532.7247314453125
}
[INFO] Day vs night summary:
   period      rmse       mae  mape_pct  p95_abs_error  count
  daytime 53.397277 22.247229 33.789715     127.711462  94608
nighttime 69.050135 37.559206 16.272956     156.918535  94608
[INFO] Top 10 biggest errors:
 sample_idx      forecast_start           timestamp  hour  horizon     actual  predicted      error  abs_error    ape_pct  is_daytime
        988 2025-03-18 17:00:00 2025-03-19 03:00:00     3       11 217.850006 750.574768 532.724762 532.724762 244.537410       False
        989 2025-03-18 18:00:00 2025-03-19 04:00:00     4       11 226.270004 755.671692 529.401688 529.401688 233.969

## Run Error Analysis Jawa


In [5]:
result = analyze_region(region_name="Jawa")
result["global_metrics"]


[INFO] Running error analysis for region: Jawa
[INFO] Using artifacts root: artifacts_xgboost_representative_points
[INFO] Using artifacts root: artifacts_xgboost_representative_points
[INFO] Global metrics:
{
  "rmse": 61.721920013427734,
  "mae": 29.90321922302246,
  "r2": 0.9464049935340881,
  "mape_pct": 23.364572525024414,
  "p90_abs_error": 100.4588623046875,
  "p95_abs_error": 144.45993041992188,
  "p99_abs_error": 245.39511108398438,
  "max_abs_error": 532.7247314453125
}
[INFO] Day vs night summary:
   period      rmse       mae  mape_pct  p95_abs_error  count
  daytime 53.397277 22.247229 33.789715     127.711462  94608
nighttime 69.050135 37.559206 16.272956     156.918535  94608
[INFO] Top 10 biggest errors:
 sample_idx      forecast_start           timestamp  hour  horizon     actual  predicted      error  abs_error    ape_pct  is_daytime
        988 2025-03-18 17:00:00 2025-03-19 03:00:00     3       11 217.850006 750.574768 532.724762 532.724762 244.537410       False
  

{'rmse': 61.721920013427734,
 'mae': 29.90321922302246,
 'r2': 0.9464049935340881,
 'mape_pct': 23.364572525024414,
 'p90_abs_error': 100.4588623046875,
 'p95_abs_error': 144.45993041992188,
 'p99_abs_error': 245.39511108398438,
 'max_abs_error': 532.7247314453125}